# Recycling model results

In [1]:
import logging
#logging.getLogger("imperative_model").setLevel(logging.DEBUG)
#logging.basicConfig(level=logging.DEBUG)

In [2]:
%load_ext autoreload
%autoreload 2
%run load_model.py

Check that processes and objects have been loaded correctly:

In [3]:
model.processes

(Process(id='Disassembly', produces=['OtherParts', 'PCBs'], consumes=['EoLTechnology'], has_stock=False),
 Process(id='EoLProcessing', produces=['EoLTechnology'], consumes=[], has_stock=False),
 Process(id='GoldAndPlasticProcessing', produces=['MixedPCBWaste', 'PureGold'], consumes=['GoldAndPlastic'], has_stock=False),
 Process(id='PCBProcess1', produces=['MixedPCBWaste', 'PureGold'], consumes=['PCBs'], has_stock=False),
 Process(id='PCBProcess2', produces=['GoldAndPlastic', 'MixedPCBWaste'], consumes=['PCBs'], has_stock=False))

In [4]:
model.objects

(Object(id='EoLTechnology', metric=rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Mass'), has_market=True),
 Object(id='GoldAndPlastic', metric=rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Mass'), has_market=True),
 Object(id='MixedPCBWaste', metric=rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Mass'), has_market=False),
 Object(id='OtherParts', metric=rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Mass'), has_market=False),
 Object(id='PCBs', metric=rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Mass'), has_market=True),
 Object(id='PureGold', metric=rdflib.term.URIRef('http://qudt.org/vocab/quantitykind/Mass'), has_market=False))

In [5]:
builder.push_process_input("Disassembly", "EoLTechnology", 7, until_objects={"PCBs"})

AdditionalActivity(values={X[0]: x2/U[0, 0], Y[0]: x2/U[0, 0]}, intermediates=[(x2, 7, 'push_process_input value of EoLTechnology from Disassembly')], transformations=[], description=None)

We can now see the parametrised solution for all the flows in the system:

In [6]:
flows = model.to_flows({eol_flow: 10, process1_gold_capacity: 100})
flows

,source,target,material,metric,value
0,Disassembly,OtherParts,OtherParts,http://qudt.org/vocab/quantitykind/Mass,8.00000000000000
1,Disassembly,PCBs,PCBs,http://qudt.org/vocab/quantitykind/Mass,2.00000000000000
2,EoLProcessing,EoLTechnology,EoLTechnology,http://qudt.org/vocab/quantitykind/Mass,0
3,GoldAndPlasticProcessing,MixedPCBWaste,MixedPCBWaste,http://qudt.org/vocab/quantitykind/Mass,0
4,GoldAndPlasticProcessing,PureGold,PureGold,http://qudt.org/vocab/quantitykind/Mass,0
5,PCBProcess1,MixedPCBWaste,MixedPCBWaste,http://qudt.org/vocab/quantitykind/Mass,"0.9*Piecewise((0, C <= 0), (2.0, C >= 0.2), (1..."
6,PCBProcess1,PureGold,PureGold,http://qudt.org/vocab/quantitykind/Mass,"0.1*Piecewise((0, C <= 0), (2.0, C >= 0.2), (1..."
7,PCBProcess2,GoldAndPlastic,GoldAndPlastic,http://qudt.org/vocab/quantitykind/Mass,0
8,PCBProcess2,MixedPCBWaste,MixedPCBWaste,http://qudt.org/vocab/quantitykind/Mass,0
9,EoLTechnology,Disassembly,EoLTechnology,http://qudt.org/vocab/quantitykind/Mass,10


In [9]:
from ipysankeywidget import SankeyWidget
from ipywidgets import Layout

links = flows.to_dict(orient='records')
links = [{"value": link["value"].subs({sy.Symbol("C"): 100}), **link} for link in links]
w = SankeyWidget(links=links, layout=Layout(width="1000", height="300"))
w.order = [
    ["EoLTechnology"],
    ["Disassembly"],
    ["PCBs", "OtherParts"],
    ["PCBProcess1", "PCBProcess2"],
    ["PureGold", "MixedPCBWaste"],
]
w

ValueError: Can't clean for JSON: 0.9*Piecewise((0, C <= 0), (2.0, C >= 0.2), (10.0*C, True))

In [8]:
%debug

> /Users/rcl38/work/probs-lab/flowprog/.venv/lib/python3.12/site-packages/jupyter_client/jsonutil.py(196)json_clean()
    192     if isinstance(obj, datetime | date):
    193         return obj.strftime(ISO8601)
    194 
    195     # we don't understand it, it's probably an unserializable object
--> 196     raise ValueError("Can't clean for JSON: %r" % obj)



ipdb>  up


> /Users/rcl38/work/probs-lab/flowprog/.venv/lib/python3.12/site-packages/jupyter_client/jsonutil.py(189)json_clean()
    187         out = {}
    188         for k, v in obj.items():
--> 189             out[str(k)] = json_clean(v)
    190         return out
    191 



ipdb>  p v


0.9*Piecewise((0, C <= 0), (2.0, C >= 0.2), (10.0*C, True))


...

ipdb>  p v.subs({sy.Symbol("C"): 1})


*** NameError: name 'sy' is not defined


ipdb>  import sympy as sy
ipdb>  p v.subs({sy.Symbol("C"): 1})


1.80000000000000


ipdb>  q


In [ ]:
links

In the interactive notebook version, you can adjust the model parameters below and see how the Sankey diagram above is affected:

In [ ]:
from ipywidgets import interact

@interact(d1=(0., 10.), d2=(0., 10.), s1=(0., 30.), a1=(0., 1.))
def calc_flows(d1=5.0, d2=2.0, s1=2.0, a1=0.5):
    flows = solution_energy_density(d1, d2, s1, a1)
    w.links = flows.to_dict(orient='records')

## Model definition history

To check how the model has been built, we can look at each variable and see what its equation looks like, with the "history" describing the steps in `load_model.py` which led to this:

In [ ]:
from IPython.display import display, Markdown

In [ ]:
Markdown(builder.describe())